# DeepGain — Training f, g, r Networks\n\nTrain three neural network components from workout logs:\n- **f** (fatigue): how much MPC drops per muscle after a set\n- **g** (RPE predictor): predicted RPE from current MPC state\n- **r** (recovery): how MPC recovers over time\n\nOnly signal: RPE prediction error. MPC is latent.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# --- Hyperparameters ---
EMBED_DIM = 16
HIDDEN_DIM = 64
LR = 1e-3
EPOCHS = 50
CHUNK_LEN = 256
BATCH_SIZE = 16
WEIGHT_SCALE = 200.0
REPS_SCALE = 30.0
DT_SCALE = np.log1p(168.0)

# --- Muscles (16 groups) ---
ALL_MUSCLES = [
    "chest", "anterior_delts", "lateral_delts", "rear_delts",
    "upper_traps", "rhomboids", "triceps", "biceps", "brachialis",
    "lats", "quads", "hamstrings", "glutes", "adductors", "erectors", "calves",
]
NUM_MUSCLES = len(ALL_MUSCLES)
MUSCLE_TO_IDX = {m: i for i, m in enumerate(ALL_MUSCLES)}

# --- Exercises (25) with muscle involvement coefficients ---
EXERCISE_MUSCLES = {
    "bench_press":        {"chest": 0.85, "triceps": 0.55, "anterior_delts": 0.60},
    "incline_bench":      {"chest": 0.70, "anterior_delts": 0.75, "triceps": 0.50},
    "close_grip_bench":   {"chest": 0.65, "triceps": 0.75, "anterior_delts": 0.55},
    "dumbbell_bench":     {"chest": 0.82, "triceps": 0.45, "anterior_delts": 0.55},
    "ohp":                {"anterior_delts": 0.85, "triceps": 0.65, "chest": 0.20, "upper_traps": 0.40},
    "dumbbell_ohp":       {"anterior_delts": 0.80, "triceps": 0.60, "upper_traps": 0.35},
    "dips":               {"chest": 0.70, "triceps": 0.65, "anterior_delts": 0.45},
    "barbell_row":        {"lats": 0.80, "biceps": 0.55, "rear_delts": 0.50, "erectors": 0.40, "upper_traps": 0.35, "rhomboids": 0.45},
    "lat_pulldown":       {"lats": 0.75, "biceps": 0.50, "rear_delts": 0.35, "rhomboids": 0.40},
    "cable_row":          {"lats": 0.70, "biceps": 0.45, "rear_delts": 0.40, "rhomboids": 0.50, "upper_traps": 0.30},
    "pull_up":            {"lats": 0.82, "biceps": 0.55, "rear_delts": 0.35, "rhomboids": 0.40},
    "squat":              {"quads": 0.85, "glutes": 0.60, "hamstrings": 0.35, "erectors": 0.45, "adductors": 0.40},
    "front_squat":        {"quads": 0.90, "glutes": 0.50, "erectors": 0.55, "adductors": 0.35},
    "deadlift":           {"glutes": 0.70, "hamstrings": 0.55, "erectors": 0.80, "quads": 0.40, "upper_traps": 0.50, "lats": 0.30, "adductors": 0.35},
    "rdl":                {"hamstrings": 0.80, "glutes": 0.55, "erectors": 0.50, "adductors": 0.25},
    "leg_press":          {"quads": 0.80, "glutes": 0.50, "adductors": 0.35},
    "bulgarian_split_squat": {"quads": 0.80, "glutes": 0.65, "hamstrings": 0.30, "adductors": 0.40},
    "hip_thrust":         {"glutes": 0.85, "hamstrings": 0.40, "adductors": 0.30},
    "tricep_pushdown":    {"triceps": 0.90},
    "overhead_tricep_ext": {"triceps": 0.85},
    "bicep_curl":         {"biceps": 0.90},
    "hammer_curl":        {"biceps": 0.75, "brachialis": 0.60},
    "lateral_raise":      {"lateral_delts": 0.85, "upper_traps": 0.30},
    "face_pull":          {"rear_delts": 0.70, "upper_traps": 0.40, "rhomboids": 0.35},
    "leg_curl":           {"hamstrings": 0.85},
    "leg_extension":      {"quads": 0.85},
    "calf_raise":         {"calves": 0.90},
}

ALL_EXERCISES = list(EXERCISE_MUSCLES.keys())
NUM_EXERCISES = len(ALL_EXERCISES)
EXERCISE_TO_IDX = {e: i for i, e in enumerate(ALL_EXERCISES)}

# Build involvement matrix (NUM_EXERCISES x NUM_MUSCLES)
INVOLVEMENT_MATRIX = np.zeros((NUM_EXERCISES, NUM_MUSCLES), dtype=np.float32)
for ex_name, muscles in EXERCISE_MUSCLES.items():
    ei = EXERCISE_TO_IDX[ex_name]
    for m_name, coeff in muscles.items():
        mi = MUSCLE_TO_IDX[m_name]
        INVOLVEMENT_MATRIX[ei, mi] = coeff

print(f"Exercises: {NUM_EXERCISES}, Muscles: {NUM_MUSCLES}")
print(f"Involvement matrix shape: {INVOLVEMENT_MATRIX.shape}")

In [ ]:
# --- Cell 2: Data Loading & 70/30 Split ---

df = pd.read_csv("training_data.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"], format="ISO8601")
df = df.sort_values(["user_id", "timestamp"]).reset_index(drop=True)
df["exercise_idx"] = df["exercise"].map(EXERCISE_TO_IDX)

# Filter out any exercises not in our mapping
df = df.dropna(subset=["exercise_idx"])
df["exercise_idx"] = df["exercise_idx"].astype(int)

# Compute delta_t in hours between consecutive sets per user
df["delta_t_hours"] = df.groupby("user_id")["timestamp"].diff().dt.total_seconds() / 3600.0
df["delta_t_hours"] = df["delta_t_hours"].fillna(0.0)

# 70/30 split by user_id
user_ids = df["user_id"].unique()
rng = np.random.RandomState(42)
rng.shuffle(user_ids)
split = int(0.7 * len(user_ids))
train_users = set(user_ids[:split])
test_users = set(user_ids[split:])

train_df = df[df["user_id"].isin(train_users)].copy()
test_df = df[df["user_id"].isin(test_users)].copy()

print(f"Total sets: {len(df):,}")
print(f"Train: {len(train_df):,} sets from {len(train_users)} users")
print(f"Test:  {len(test_df):,} sets from {len(test_users)} users")

In [ ]:
# --- Cell 3: Dataset & DataLoader ---

def build_user_sequences(user_df):
    """Convert a DataFrame into a list of per-user sequence dicts."""
    sequences = []
    for uid, grp in user_df.groupby("user_id"):
        seq = {
            "user_id": uid,
            "exercise_idx": torch.tensor(grp["exercise_idx"].values, dtype=torch.long),
            "weight": torch.tensor(grp["weight"].values / WEIGHT_SCALE, dtype=torch.float32),
            "reps": torch.tensor(grp["reps"].values / REPS_SCALE, dtype=torch.float32),
            "rpe": torch.tensor(grp["rpe"].values / 10.0, dtype=torch.float32),
            "delta_t": torch.tensor(
                np.log1p(grp["delta_t_hours"].values) / DT_SCALE, dtype=torch.float32
            ),
            "timestamps": grp["timestamp"].values,
        }
        sequences.append(seq)
    return sequences

def chunk_sequence(seq, chunk_len):
    """Split a user sequence into chunks of chunk_len."""
    T = len(seq["exercise_idx"])
    if T <= chunk_len:
        return [seq]
    chunks = []
    for start in range(0, T, chunk_len):
        end = min(start + chunk_len, T)
        chunk = {
            "user_id": seq["user_id"],
            "exercise_idx": seq["exercise_idx"][start:end],
            "weight": seq["weight"][start:end],
            "reps": seq["reps"][start:end],
            "rpe": seq["rpe"][start:end],
            "delta_t": seq["delta_t"][start:end],
        }
        if "timestamps" in seq:
            chunk["timestamps"] = seq["timestamps"][start:end]
        chunks.append(chunk)
    return chunks

class ChunkedDataset(Dataset):
    def __init__(self, user_df, chunk_len):
        sequences = build_user_sequences(user_df)
        self.chunks = []
        for seq in sequences:
            self.chunks.extend(chunk_sequence(seq, chunk_len))

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        return self.chunks[idx]

def collate_fn(batch):
    max_len = max(len(b["exercise_idx"]) for b in batch)
    B = len(batch)
    out = {
        "exercise_idx": torch.zeros(B, max_len, dtype=torch.long),
        "weight": torch.zeros(B, max_len),
        "reps": torch.zeros(B, max_len),
        "rpe": torch.zeros(B, max_len),
        "delta_t": torch.zeros(B, max_len),
        "mask": torch.zeros(B, max_len),
    }
    for i, b in enumerate(batch):
        T = len(b["exercise_idx"])
        out["exercise_idx"][i, :T] = b["exercise_idx"]
        out["weight"][i, :T] = b["weight"]
        out["reps"][i, :T] = b["reps"]
        out["rpe"][i, :T] = b["rpe"]
        out["delta_t"][i, :T] = b["delta_t"]
        out["mask"][i, :T] = 1.0
    return out

train_ds = ChunkedDataset(train_df, CHUNK_LEN)
test_ds = ChunkedDataset(test_df, CHUNK_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train chunks: {len(train_ds)}, Test chunks: {len(test_ds)}")
print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

In [ ]:
# --- Cell 4: Model Architecture ---

class FatigueNet(nn.Module):
    """f: predicts MPC drop fraction after a set."""
    def __init__(self, embed_dim, hidden_dim):
        super().__init__()
        # inputs: weight(1) + reps(1) + rpe(1) + current_mpc(1) + exercise_embed + muscle_embed
        self.net = nn.Sequential(
            nn.Linear(4 + 2 * embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),  # drop fraction in (0, 1)
        )
        # Initialize output bias small so initial drops are gentle
        nn.init.constant_(self.net[-2].bias, -2.0)

    def forward(self, weight, reps, rpe, mpc, e_embed, m_embed):
        # All inputs: (N,) scalars or (N, E) embeddings
        x = torch.cat([
            weight.unsqueeze(-1), reps.unsqueeze(-1),
            rpe.unsqueeze(-1), mpc.unsqueeze(-1),
            e_embed, m_embed
        ], dim=-1)
        return self.net(x).squeeze(-1)


class RPENet(nn.Module):
    """g: predicts RPE from current MPC state."""
    def __init__(self, embed_dim, hidden_dim, num_muscles):
        super().__init__()
        # inputs: weight(1) + reps(1) + exercise_embed(E) + all_mpc(16)
        self.net = nn.Sequential(
            nn.Linear(2 + embed_dim + num_muscles, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, weight, reps, e_embed, mpc_all):
        # weight, reps: (B,), e_embed: (B, E), mpc_all: (B, M)
        x = torch.cat([
            weight.unsqueeze(-1), reps.unsqueeze(-1),
            e_embed, mpc_all
        ], dim=-1)
        raw = self.net(x).squeeze(-1)
        return torch.sigmoid(raw)  # output in (0, 1), will scale to RPE [0, 1] normalized


class RecoveryNet(nn.Module):
    """r: predicts recovery fraction toward MPC=1.0."""
    def __init__(self, embed_dim, hidden_dim):
        super().__init__()
        # inputs: current_mpc(1) + delta_t(1) + muscle_embed(E)
        self.net = nn.Sequential(
            nn.Linear(2 + embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, mpc, delta_t, m_embed):
        # mpc, delta_t: (N,), m_embed: (N, E)
        x = torch.cat([mpc.unsqueeze(-1), delta_t.unsqueeze(-1), m_embed], dim=-1)
        recovery_frac = torch.sigmoid(self.net(x).squeeze(-1))
        # MPC recovers toward 1.0: new_mpc = mpc + (1 - mpc) * recovery_fraction
        return mpc + (1.0 - mpc) * recovery_frac


class DeepGainModel(nn.Module):
    def __init__(self, num_exercises, num_muscles, embed_dim, hidden_dim):
        super().__init__()
        self.num_muscles = num_muscles
        self.exercise_embed = nn.Embedding(num_exercises, embed_dim)
        self.muscle_embed = nn.Embedding(num_muscles, embed_dim)
        self.f_net = FatigueNet(embed_dim, hidden_dim)
        self.g_net = RPENet(embed_dim, hidden_dim, num_muscles)
        self.r_net = RecoveryNet(embed_dim, hidden_dim)
        self.register_buffer("involvement",
                             torch.tensor(INVOLVEMENT_MATRIX, dtype=torch.float32))

    def forward(self, exercise_idx, weight, reps, rpe_target, delta_t, mask):
        B, T = exercise_idx.shape
        M = self.num_muscles
        device = exercise_idx.device

        mpc = torch.ones(B, M, device=device)
        rpe_preds = []

        # Pre-compute muscle embeddings (shared across timesteps)
        all_m_idx = torch.arange(M, device=device)
        all_m_embed = self.muscle_embed(all_m_idx)  # (M, E)
        all_m_embed_expanded = all_m_embed.unsqueeze(0).expand(B, -1, -1)  # (B, M, E)
        E = all_m_embed.shape[-1]

        for t in range(T):
            # 1. Recovery (vectorized over muscles)
            if t > 0:
                dt = delta_t[:, t]  # (B,)
                dt_exp = dt.unsqueeze(1).expand(-1, M).reshape(-1)  # (B*M,)
                mpc_flat = mpc.reshape(-1)  # (B*M,)
                m_emb_flat = all_m_embed_expanded.reshape(-1, E)  # (B*M, E)
                mpc = self.r_net(mpc_flat, dt_exp, m_emb_flat).reshape(B, M)

            # 2. RPE prediction
            e_idx = exercise_idx[:, t]
            e_embed = self.exercise_embed(e_idx)  # (B, E)
            rpe_pred = self.g_net(weight[:, t], reps[:, t], e_embed, mpc)
            rpe_preds.append(rpe_pred)

            # 3. Fatigue update (vectorized over muscles)
            inv = self.involvement[e_idx]  # (B, M)
            e_emb_exp = e_embed.unsqueeze(1).expand(-1, M, -1).reshape(-1, E)  # (B*M, E)
            w_exp = weight[:, t].unsqueeze(1).expand(-1, M).reshape(-1)
            r_exp = reps[:, t].unsqueeze(1).expand(-1, M).reshape(-1)
            rpe_exp = rpe_target[:, t].unsqueeze(1).expand(-1, M).reshape(-1)
            mpc_flat = mpc.reshape(-1)
            m_emb_flat = all_m_embed_expanded.reshape(-1, E)

            drop = self.f_net(w_exp, r_exp, rpe_exp, mpc_flat, e_emb_exp, m_emb_flat)
            drop = drop.reshape(B, M)
            mpc = (mpc * (1.0 - inv * drop)).clamp(min=0.1)

        rpe_preds = torch.stack(rpe_preds, dim=1)  # (B, T)
        return rpe_preds, mpc

    def forward_with_mpc_history(self, exercise_idx, weight, reps, rpe_target, delta_t):
        """Single sequence forward pass that records MPC at every step."""
        B, T = exercise_idx.shape
        assert B == 1, "Use batch size 1 for MPC history"
        M = self.num_muscles
        device = exercise_idx.device

        mpc = torch.ones(1, M, device=device)
        rpe_preds = []
        mpc_history = [mpc[0].detach().cpu().numpy().copy()]

        all_m_idx = torch.arange(M, device=device)
        all_m_embed = self.muscle_embed(all_m_idx)
        all_m_embed_expanded = all_m_embed.unsqueeze(0)
        E = all_m_embed.shape[-1]

        for t in range(T):
            if t > 0:
                dt = delta_t[:, t]
                dt_exp = dt.unsqueeze(1).expand(-1, M).reshape(-1)
                mpc_flat = mpc.reshape(-1)
                m_emb_flat = all_m_embed_expanded.reshape(-1, E)
                mpc = self.r_net(mpc_flat, dt_exp, m_emb_flat).reshape(1, M)

            e_idx = exercise_idx[:, t]
            e_embed = self.exercise_embed(e_idx)
            rpe_pred = self.g_net(weight[:, t], reps[:, t], e_embed, mpc)
            rpe_preds.append(rpe_pred.item())

            inv = self.involvement[e_idx]
            e_emb_exp = e_embed.unsqueeze(1).expand(-1, M, -1).reshape(-1, E)
            w_exp = weight[:, t].unsqueeze(1).expand(-1, M).reshape(-1)
            r_exp = reps[:, t].unsqueeze(1).expand(-1, M).reshape(-1)
            rpe_exp = rpe_target[:, t].unsqueeze(1).expand(-1, M).reshape(-1)
            mpc_flat = mpc.reshape(-1)
            m_emb_flat = all_m_embed_expanded.reshape(-1, E)
            drop = self.f_net(w_exp, r_exp, rpe_exp, mpc_flat, e_emb_exp, m_emb_flat).reshape(1, M)
            mpc = (mpc * (1.0 - inv * drop)).clamp(min=0.1)
            mpc_history.append(mpc[0].detach().cpu().numpy().copy())

        return np.array(rpe_preds), np.array(mpc_history)

model = DeepGainModel(NUM_EXERCISES, NUM_MUSCLES, EMBED_DIM, HIDDEN_DIM).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(model)

In [ ]:
# --- Cell 5: Load Pretrained Model ---
# Model was trained via train.py (50 epochs, RMSE 1.01, MAE 0.70, R=0.90)

model.load_state_dict(torch.load("deepgain_model.pt", map_location=DEVICE))
model.eval()
print("Loaded pretrained model from deepgain_model.pt")

In [ ]:
# --- Cell 6: Evaluation Metrics ---

@torch.no_grad()
def collect_predictions(model, loader):
    """Collect all predictions and targets from a dataloader."""
    model.eval()
    all_preds = []
    all_targets = []
    all_exercises = []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        rpe_pred, _ = model(batch["exercise_idx"], batch["weight"],
                            batch["reps"], batch["rpe"], batch["delta_t"], batch["mask"])
        mask = batch["mask"].bool()
        all_preds.append((rpe_pred[mask] * 10.0).cpu().numpy())
        all_targets.append((batch["rpe"][mask] * 10.0).cpu().numpy())
        all_exercises.append(batch["exercise_idx"][mask].cpu().numpy())

    preds = np.concatenate(all_preds)
    targets = np.concatenate(all_targets)
    exercises = np.concatenate(all_exercises)
    return preds, targets, exercises

test_preds, test_targets, test_exercises = collect_predictions(model, test_loader)

mse = np.mean((test_preds - test_targets) ** 2)
mae = np.mean(np.abs(test_preds - test_targets))
rmse = np.sqrt(mse)
corr = np.corrcoef(test_preds, test_targets)[0, 1]

print(f"Test Set Metrics (RPE scale 1-10):")
print(f"  MSE:  {mse:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"  Correlation: {corr:.4f}")
print(f"  N predictions: {len(test_preds):,}")

# Per-exercise MAE
exercise_maes = {}
for ei in range(NUM_EXERCISES):
    mask = test_exercises == ei
    if mask.sum() > 0:
        exercise_maes[ALL_EXERCISES[ei]] = np.mean(np.abs(test_preds[mask] - test_targets[mask]))

print(f"\nPer-exercise MAE:")
for ex, mae_val in sorted(exercise_maes.items(), key=lambda x: x[1]):
    print(f"  {ex:25s}: {mae_val:.3f}")

In [ ]:
# --- Chart: Training Loss Curves ---

train_rmses = [2.08, 1.50, 1.36, 1.31, 1.28, 1.26, 1.25, 1.23, 1.21, 1.20,
               1.19, 1.18, 1.17, 1.16, 1.15, 1.14, 1.13, 1.11, 1.11, 1.10,
               1.09, 1.08, 1.08, 1.07, 1.07, 1.06, 1.06, 1.05, 1.05, 1.04,
               1.04, 1.04, 1.03, 1.03, 1.02, 1.03, 1.02, 1.02, 1.02, 1.02,
               1.01, 1.01, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 0.99, 0.99]
val_rmses =   [1.64, 1.44, 1.34, 1.31, 1.28, 1.28, 1.27, 1.27, 1.24, 1.24,
               1.23, 1.19, 1.19, 1.29, 1.16, 1.14, 1.13, 1.21, 1.15, 1.11,
               1.10, 1.09, 1.11, 1.10, 1.09, 1.13, 1.09, 1.06, 1.07, 1.07,
               1.05, 1.06, 1.09, 1.05, 1.07, 1.04, 1.05, 1.06, 1.05, 1.04,
               1.06, 1.06, 1.01, 1.02, 1.02, 1.02, 1.02, 1.02, 1.01, 1.01]

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(range(1, 51), train_rmses, label="Train RMSE", linewidth=2)
ax.plot(range(1, 51), val_rmses, label="Val RMSE", linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("RMSE (RPE scale)")
ax.set_title("Training & Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Cell 8: RPE Prediction Accuracy ---

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Scatter: predicted vs actual
ax = axes[0]
n_sample = min(15000, len(test_preds))
idx = np.random.choice(len(test_preds), n_sample, replace=False)
ax.scatter(test_targets[idx], test_preds[idx], alpha=0.05, s=8, c="steelblue")
ax.plot([1, 10], [1, 10], "r--", linewidth=2, label="Perfect")
r2 = corr ** 2
ax.set_xlabel("Actual RPE")
ax.set_ylabel("Predicted RPE")
ax.set_title(f"Predicted vs Actual RPE (R²={r2:.3f})")
ax.legend()
ax.set_xlim(1, 10.5)
ax.set_ylim(1, 10.5)
ax.grid(True, alpha=0.3)

# 2. Per-exercise MAE bar chart
ax = axes[1]
sorted_exercises = sorted(exercise_maes.items(), key=lambda x: x[1], reverse=True)
names = [e[0].replace("_", " ") for e in sorted_exercises]
vals = [e[1] for e in sorted_exercises]
colors = ["#e74c3c" if v > np.median(vals) else "#2ecc71" for v in vals]
ax.barh(range(len(names)), vals, color=colors, alpha=0.8)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=8)
ax.set_xlabel("MAE (RPE)")
ax.set_title("Per-Exercise MAE")
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis="x")

# 3. Error distribution
ax = axes[2]
errors = test_preds - test_targets
ax.hist(errors, bins=80, color="steelblue", alpha=0.7, density=True)
ax.axvline(0, color="red", linestyle="--", linewidth=2)
ax.set_xlabel("RPE Error (pred - actual)")
ax.set_ylabel("Density")
ax.set_title(f"Error Distribution (mean={np.mean(errors):.3f}, std={np.std(errors):.3f})")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- Cell 9: MPC Trajectories Over Time (3 sample test users) ---

test_sequences = build_user_sequences(test_df)
# Pick 3 users with decent sequence length
sample_users = sorted(test_sequences, key=lambda s: len(s["exercise_idx"]), reverse=True)[:3]

model.eval()
fig, axes = plt.subplots(3, 2, figsize=(20, 18))

# Muscle colors for consistent plotting
muscle_colors = plt.cm.tab20(np.linspace(0, 1, NUM_MUSCLES))

for row, seq in enumerate(sample_users):
    uid = seq["user_id"]
    T = len(seq["exercise_idx"])
    # Limit to first 400 sets for readability
    T_plot = min(T, 400)

    with torch.no_grad():
        ex = seq["exercise_idx"][:T_plot].unsqueeze(0).to(DEVICE)
        w = seq["weight"][:T_plot].unsqueeze(0).to(DEVICE)
        r = seq["reps"][:T_plot].unsqueeze(0).to(DEVICE)
        rpe = seq["rpe"][:T_plot].unsqueeze(0).to(DEVICE)
        dt = seq["delta_t"][:T_plot].unsqueeze(0).to(DEVICE)
        rpe_preds_seq, mpc_hist = model.forward_with_mpc_history(ex, w, r, rpe, dt)

    timestamps = seq["timestamps"][:T_plot]
    # Convert to hours from start for x-axis
    t0 = timestamps[0]
    hours = [(t - t0) / np.timedelta64(1, "h") for t in timestamps]

    # MPC plot
    ax = axes[row, 0]
    # Find muscles that were actually used by this user
    used_exercises = seq["exercise_idx"][:T_plot].numpy()
    used_muscles = set()
    for ei in used_exercises:
        for mi in range(NUM_MUSCLES):
            if INVOLVEMENT_MATRIX[ei, mi] > 0:
                used_muscles.add(mi)
    # Plot top 6 most active muscles
    muscle_activity = {}
    for mi in used_muscles:
        muscle_activity[mi] = np.sum(INVOLVEMENT_MATRIX[used_exercises, mi])
    top_muscles = sorted(muscle_activity, key=muscle_activity.get, reverse=True)[:6]

    for mi in top_muscles:
        ax.plot(hours, mpc_hist[:-1, mi], label=ALL_MUSCLES[mi].replace("_", " "),
                alpha=0.8, linewidth=1.2, color=muscle_colors[mi])
    ax.set_ylabel("MPC")
    ax.set_title(f"MPC Over Time — {uid} ({T_plot} sets)")
    ax.legend(fontsize=7, loc="lower left")
    ax.set_ylim(0.0, 1.05)
    ax.grid(True, alpha=0.3)
    if row == 2:
        ax.set_xlabel("Hours from first set")

    # RPE plot
    ax = axes[row, 1]
    actual_rpe = seq["rpe"][:T_plot].numpy() * 10.0
    ax.scatter(hours, actual_rpe, s=8, alpha=0.4, c="steelblue", label="Actual RPE")
    ax.scatter(hours, rpe_preds_seq * 10.0, s=8, alpha=0.4, c="coral", label="Predicted RPE")
    ax.set_ylabel("RPE")
    ax.set_title(f"RPE Prediction — {uid}")
    ax.legend(fontsize=8)
    ax.set_ylim(1, 11)
    ax.grid(True, alpha=0.3)
    if row == 2:
        ax.set_xlabel("Hours from first set")

plt.tight_layout()
plt.show()

In [ ]:
# --- Cell 10: Learned Recovery Curves ---
# Probe the recovery network: for each muscle, sweep delta_t from 0 to 72h
# starting from MPC = 0.5

model.eval()
dt_hours = np.linspace(0, 72, 200)
dt_normalized = np.log1p(dt_hours) / DT_SCALE

fig, ax = plt.subplots(figsize=(12, 6))

with torch.no_grad():
    for mi in range(NUM_MUSCLES):
        m_embed = model.muscle_embed(torch.tensor([mi], device=DEVICE))  # (1, E)
        m_embed_exp = m_embed.expand(len(dt_normalized), -1)  # (N, E)

        mpc_start = torch.full((len(dt_normalized),), 0.5, device=DEVICE)
        dt_tensor = torch.tensor(dt_normalized, dtype=torch.float32, device=DEVICE)

        recovered = model.r_net(mpc_start, dt_tensor, m_embed_exp).cpu().numpy()

        ax.plot(dt_hours, recovered, label=ALL_MUSCLES[mi].replace("_", " "),
                color=muscle_colors[mi], linewidth=1.5, alpha=0.8)

ax.axhline(y=1.0, color="gray", linestyle=":", alpha=0.5)
ax.axhline(y=0.5, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("Time (hours)")
ax.set_ylabel("MPC")
ax.set_title("Learned Recovery Curves (starting MPC = 0.5)")
ax.legend(fontsize=7, ncol=4, loc="lower right")
ax.set_ylim(0.4, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Cell 11: Fatigue Heatmaps ---
# Probe the fatigue network: sweep weight x reps for 3 exercises

model.eval()
probe_exercises = ["bench_press", "squat", "bicep_curl"]
weight_range = np.linspace(0.1, 1.0, 30)  # normalized (20-200 kg)
reps_range = np.linspace(0.03, 0.5, 30)   # normalized (1-15 reps)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

with torch.no_grad():
    for col, ex_name in enumerate(probe_exercises):
        ei = EXERCISE_TO_IDX[ex_name]
        # Find primary muscle for this exercise
        muscles = EXERCISE_MUSCLES[ex_name]
        primary_muscle = max(muscles, key=muscles.get)
        mi = MUSCLE_TO_IDX[primary_muscle]

        e_embed = model.exercise_embed(torch.tensor([ei], device=DEVICE))  # (1, E)
        m_embed = model.muscle_embed(torch.tensor([mi], device=DEVICE))  # (1, E)

        drop_grid = np.zeros((len(reps_range), len(weight_range)))

        for ri, r_val in enumerate(reps_range):
            for wi, w_val in enumerate(weight_range):
                w_t = torch.tensor([w_val], dtype=torch.float32, device=DEVICE)
                r_t = torch.tensor([r_val], dtype=torch.float32, device=DEVICE)
                rpe_t = torch.tensor([0.8], dtype=torch.float32, device=DEVICE)  # RPE 8
                mpc_t = torch.tensor([1.0], dtype=torch.float32, device=DEVICE)

                d = model.f_net(w_t, r_t, rpe_t, mpc_t, e_embed, m_embed).item()
                drop_grid[ri, wi] = d * muscles[primary_muscle]  # effective drop

        ax = axes[col]
        im = ax.imshow(drop_grid, aspect="auto", origin="lower",
                       extent=[20, 200, 1, 15],
                       cmap="YlOrRd", vmin=0)
        ax.set_xlabel("Weight (kg)")
        ax.set_ylabel("Reps")
        ax.set_title(f"MPC Drop: {ex_name.replace('_', ' ')}\n(primary: {primary_muscle})")
        plt.colorbar(im, ax=ax, label="MPC drop")

plt.suptitle("Fatigue Response (f) at RPE 8, MPC=1.0", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Chart: Exercise Embedding Similarity ---
# Which exercises does the model consider similar?

model.eval()
with torch.no_grad():
    ex_embeds = model.exercise_embed.weight.cpu().numpy()  # (27, 16)
    m_embeds = model.muscle_embed.weight.cpu().numpy()     # (16, 16)

# Cosine similarity between exercises
from numpy.linalg import norm
ex_sim = np.zeros((NUM_EXERCISES, NUM_EXERCISES))
for i in range(NUM_EXERCISES):
    for j in range(NUM_EXERCISES):
        ex_sim[i, j] = np.dot(ex_embeds[i], ex_embeds[j]) / (norm(ex_embeds[i]) * norm(ex_embeds[j]) + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Exercise similarity heatmap
ax = axes[0]
ex_labels = [e.replace("_", " ") for e in ALL_EXERCISES]
im = ax.imshow(ex_sim, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(NUM_EXERCISES))
ax.set_yticks(range(NUM_EXERCISES))
ax.set_xticklabels(ex_labels, rotation=90, fontsize=7)
ax.set_yticklabels(ex_labels, fontsize=7)
ax.set_title("Exercise Embedding Cosine Similarity")
plt.colorbar(im, ax=ax, shrink=0.8)

# Muscle similarity heatmap
ax = axes[1]
m_sim = np.zeros((NUM_MUSCLES, NUM_MUSCLES))
for i in range(NUM_MUSCLES):
    for j in range(NUM_MUSCLES):
        m_sim[i, j] = np.dot(m_embeds[i], m_embeds[j]) / (norm(m_embeds[i]) * norm(m_embeds[j]) + 1e-8)

m_labels = [m.replace("_", " ") for m in ALL_MUSCLES]
im = ax.imshow(m_sim, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(NUM_MUSCLES))
ax.set_yticks(range(NUM_MUSCLES))
ax.set_xticklabels(m_labels, rotation=90, fontsize=8)
ax.set_yticklabels(m_labels, fontsize=8)
ax.set_title("Muscle Embedding Cosine Similarity")
plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

In [ ]:
# --- Chart: Per-Muscle Fatigue Breakdown per Exercise ---
# For each exercise, how much does each involved muscle fatigue from one set?

model.eval()
probe_exercises = ["bench_press", "squat", "deadlift", "ohp", "barbell_row", "dips"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

with torch.no_grad():
    for idx, ex_name in enumerate(probe_exercises):
        ax = axes[idx // 3, idx % 3]
        ei = EXERCISE_TO_IDX[ex_name]
        muscles = EXERCISE_MUSCLES[ex_name]
        e_embed = model.exercise_embed(torch.tensor([ei], device=DEVICE))

        involved = sorted(muscles.keys(), key=lambda m: muscles[m], reverse=True)
        drops = []
        colors_bar = []

        for m_name in involved:
            mi = MUSCLE_TO_IDX[m_name]
            m_embed = model.muscle_embed(torch.tensor([mi], device=DEVICE))
            # Simulate at moderate intensity: 0.4 weight (~80kg), 0.27 reps (~8), RPE 8
            w_t = torch.tensor([0.4], dtype=torch.float32, device=DEVICE)
            r_t = torch.tensor([0.27], dtype=torch.float32, device=DEVICE)
            rpe_t = torch.tensor([0.8], dtype=torch.float32, device=DEVICE)
            mpc_t = torch.tensor([1.0], dtype=torch.float32, device=DEVICE)

            raw_drop = model.f_net(w_t, r_t, rpe_t, mpc_t, e_embed, m_embed).item()
            effective_drop = raw_drop * muscles[m_name]
            drops.append(effective_drop)
            colors_bar.append(muscle_colors[MUSCLE_TO_IDX[m_name]])

        labels = [m.replace("_", " ") for m in involved]
        bars = ax.barh(range(len(labels)), drops, color=colors_bar, alpha=0.85)
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels, fontsize=9)
        ax.set_xlabel("MPC Drop per Set")
        ax.set_title(f"{ex_name.replace('_', ' ').title()}\n(80kg, 8 reps, RPE 8)")
        ax.invert_yaxis()
        ax.grid(True, alpha=0.3, axis="x")
        # Annotate involvement coefficients
        for i, (d, m_name) in enumerate(zip(drops, involved)):
            ax.text(d + 0.002, i, f"inv={muscles[m_name]:.2f}", va="center", fontsize=7, color="gray")

plt.suptitle("Learned Fatigue (f) — Per-Muscle Breakdown by Exercise", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- Chart: Simulated Push Day — MPC Evolution Set by Set ---
# Simulate a realistic push workout and track MPC drop + recovery between sets

model.eval()

# Push day: bench 4x8, OHP 3x8, dips 3x10, tricep pushdown 3x12
workout = [
    ("bench_press",     80, 8, 7.5, 0.05),   # (exercise, weight_kg, reps, rpe, rest_hours)
    ("bench_press",     80, 7, 8.0, 0.05),
    ("bench_press",     80, 6, 8.5, 0.05),
    ("bench_press",     80, 5, 9.0, 0.05),
    ("ohp",             40, 8, 7.5, 0.05),
    ("ohp",             40, 7, 8.0, 0.05),
    ("ohp",             40, 6, 8.5, 0.05),
    ("dips",            20, 10, 7.0, 0.04),
    ("dips",            20, 9, 7.5, 0.04),
    ("dips",            20, 8, 8.0, 0.04),
    ("tricep_pushdown", 25, 12, 7.0, 0.03),
    ("tricep_pushdown", 25, 11, 7.5, 0.03),
    ("tricep_pushdown", 25, 10, 8.0, 0.03),
]

mpc_history = []
rpe_preds = []
set_labels = []

with torch.no_grad():
    mpc = torch.ones(1, NUM_MUSCLES, device=DEVICE)
    all_m_idx = torch.arange(NUM_MUSCLES, device=DEVICE)
    all_m_embed = model.muscle_embed(all_m_idx).unsqueeze(0)
    E = all_m_embed.shape[-1]

    for i, (ex, w_kg, reps, rpe_actual, rest_h) in enumerate(workout):
        # Recovery from rest
        if i > 0:
            dt_norm = torch.tensor([np.log1p(rest_h) / DT_SCALE], device=DEVICE)
            dt_exp = dt_norm.unsqueeze(1).expand(-1, NUM_MUSCLES).reshape(-1)
            mpc = model.r_net(mpc.reshape(-1), dt_exp, all_m_embed.reshape(-1, E)).reshape(1, NUM_MUSCLES)

        mpc_history.append(mpc[0].detach().cpu().numpy().copy())

        ei = EXERCISE_TO_IDX[ex]
        e_embed = model.exercise_embed(torch.tensor([ei], device=DEVICE))
        w_norm = torch.tensor([w_kg / WEIGHT_SCALE], device=DEVICE)
        r_norm = torch.tensor([reps / REPS_SCALE], device=DEVICE)
        rpe_norm = torch.tensor([rpe_actual / 10.0], device=DEVICE)

        pred = model.g_net(w_norm, r_norm, e_embed, mpc).item() * 10.0
        rpe_preds.append(pred)
        set_labels.append(f"{ex.replace('_', ' ')}\n{w_kg}kg x{reps}")

        # Apply fatigue
        inv = model.involvement[torch.tensor([ei], device=DEVICE)]
        e_emb_exp = e_embed.unsqueeze(1).expand(-1, NUM_MUSCLES, -1).reshape(-1, E)
        w_exp = w_norm.unsqueeze(1).expand(-1, NUM_MUSCLES).reshape(-1)
        r_exp = r_norm.unsqueeze(1).expand(-1, NUM_MUSCLES).reshape(-1)
        rpe_exp = rpe_norm.unsqueeze(1).expand(-1, NUM_MUSCLES).reshape(-1)
        drop = model.f_net(w_exp, r_exp, rpe_exp, mpc.reshape(-1), e_emb_exp, all_m_embed.reshape(-1, E))
        drop = drop.reshape(1, NUM_MUSCLES)
        mpc = (mpc * (1.0 - inv * drop)).clamp(min=0.1)

mpc_arr = np.array(mpc_history)

fig, axes = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={"height_ratios": [2, 1]})

# MPC evolution
ax = axes[0]
push_muscles = ["chest", "anterior_delts", "triceps", "lateral_delts", "upper_traps"]
for m_name in push_muscles:
    mi = MUSCLE_TO_IDX[m_name]
    ax.plot(range(len(mpc_arr)), mpc_arr[:, mi], "o-", label=m_name.replace("_", " "),
            color=muscle_colors[mi], linewidth=2, markersize=5)

ax.set_xticks(range(len(set_labels)))
ax.set_xticklabels(set_labels, rotation=45, ha="right", fontsize=7)
ax.set_ylabel("MPC")
ax.set_title("Push Day Simulation — MPC Per Muscle (before each set)")
ax.legend(fontsize=9)
ax.set_ylim(0.3, 1.05)
ax.grid(True, alpha=0.3)

# Add exercise group shading
ax.axvspan(-0.5, 3.5, alpha=0.08, color="blue", label="Bench")
ax.axvspan(3.5, 6.5, alpha=0.08, color="green", label="OHP")
ax.axvspan(6.5, 9.5, alpha=0.08, color="orange", label="Dips")
ax.axvspan(9.5, 12.5, alpha=0.08, color="red", label="Pushdown")

# RPE predictions
ax = axes[1]
actual_rpes = [w[3] for w in workout]
ax.bar(range(len(rpe_preds)), actual_rpes, alpha=0.4, color="steelblue", label="Actual RPE", width=0.4)
ax.bar([x + 0.4 for x in range(len(rpe_preds))], rpe_preds, alpha=0.7, color="coral", label="Predicted RPE", width=0.4)
ax.set_xticks(range(len(set_labels)))
ax.set_xticklabels(set_labels, rotation=45, ha="right", fontsize=7)
ax.set_ylabel("RPE")
ax.set_title("RPE: Actual vs Predicted")
ax.legend()
ax.set_ylim(4, 10.5)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

In [ ]:
# --- Chart: Recovery Time to 90% MPC per Muscle ---
# How many hours does each muscle take to recover from MPC=0.5 to MPC=0.9?

model.eval()
dt_hours_fine = np.linspace(0, 96, 500)
dt_norm_fine = np.log1p(dt_hours_fine) / DT_SCALE

recovery_times = {}  # muscle -> hours to reach 0.9

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

with torch.no_grad():
    for mi in range(NUM_MUSCLES):
        m_embed = model.muscle_embed(torch.tensor([mi], device=DEVICE))
        m_embed_exp = m_embed.expand(len(dt_norm_fine), -1)
        mpc_start = torch.full((len(dt_norm_fine),), 0.5, device=DEVICE)
        dt_tensor = torch.tensor(dt_norm_fine, dtype=torch.float32, device=DEVICE)
        recovered = model.r_net(mpc_start, dt_tensor, m_embed_exp).cpu().numpy()

        # Find time to reach 0.9
        idx_90 = np.where(recovered >= 0.9)[0]
        if len(idx_90) > 0:
            recovery_times[ALL_MUSCLES[mi]] = dt_hours_fine[idx_90[0]]
        else:
            recovery_times[ALL_MUSCLES[mi]] = 96.0  # didn't reach in 96h

# Bar chart of recovery times
ax = axes[0]
sorted_rec = sorted(recovery_times.items(), key=lambda x: x[1], reverse=True)
names = [m.replace("_", " ") for m, _ in sorted_rec]
times = [t for _, t in sorted_rec]
bar_colors = [muscle_colors[MUSCLE_TO_IDX[m]] for m, _ in sorted_rec]
ax.barh(range(len(names)), times, color=bar_colors, alpha=0.85)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel("Hours to 90% MPC")
ax.set_title("Learned Recovery Speed (from MPC=0.5 to MPC=0.9)")
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis="x")

# Ground truth recovery tau comparison
ax = axes[1]
GROUND_TRUTH_TAU = {
    "quads": 36.0, "glutes": 32.0, "hamstrings": 28.0, "adductors": 28.0,
    "erectors": 26.0, "calves": 20.0, "chest": 22.0, "lats": 24.0,
    "anterior_delts": 20.0, "lateral_delts": 18.0, "rear_delts": 18.0,
    "upper_traps": 20.0, "rhomboids": 20.0, "triceps": 16.0,
    "biceps": 16.0, "brachialis": 16.0,
}

learned_times = [recovery_times[m] for m in ALL_MUSCLES]
gt_tau = [GROUND_TRUTH_TAU[m] for m in ALL_MUSCLES]
# Tau to 90% for exponential: t = -tau * ln(1 - 0.8) = tau * ln(5) ≈ 1.6 * tau
gt_times_90 = [t * np.log(5) for t in gt_tau]

ax.scatter(gt_times_90, learned_times, s=80, c=[muscle_colors[i] for i in range(NUM_MUSCLES)], zorder=5)
for i, m in enumerate(ALL_MUSCLES):
    ax.annotate(m.replace("_", " "), (gt_times_90[i], learned_times[i]),
                fontsize=7, ha="left", va="bottom", xytext=(4, 4), textcoords="offset points")

max_val = max(max(gt_times_90), max(learned_times)) * 1.1
ax.plot([0, max_val], [0, max_val], "r--", alpha=0.5, label="Perfect match")
ax.set_xlabel("Ground Truth Time to 90% (hours)")
ax.set_ylabel("Learned Time to 90% (hours)")
ax.set_title("Recovery: Learned vs Ground Truth")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- Chart: RPE Sensitivity to MPC — Which muscles matter most for each exercise? ---
# Probe g: hold all MPCs at 1.0 except one muscle, drop it to 0.5, measure RPE change

model.eval()
probe_exercises = ["bench_press", "squat", "deadlift", "ohp", "barbell_row", "lat_pulldown"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

with torch.no_grad():
    for idx, ex_name in enumerate(probe_exercises):
        ax = axes[idx // 3, idx % 3]
        ei = EXERCISE_TO_IDX[ex_name]
        e_embed = model.exercise_embed(torch.tensor([ei], device=DEVICE))
        w_norm = torch.tensor([0.4], device=DEVICE)
        r_norm = torch.tensor([0.27], device=DEVICE)

        # Baseline: all MPC = 1.0
        mpc_base = torch.ones(1, NUM_MUSCLES, device=DEVICE)
        rpe_base = model.g_net(w_norm, r_norm, e_embed, mpc_base).item() * 10.0

        # Drop each muscle to 0.5 one at a time
        deltas = {}
        for mi in range(NUM_MUSCLES):
            mpc_test = torch.ones(1, NUM_MUSCLES, device=DEVICE)
            mpc_test[0, mi] = 0.5
            rpe_test = model.g_net(w_norm, r_norm, e_embed, mpc_test).item() * 10.0
            deltas[ALL_MUSCLES[mi]] = rpe_test - rpe_base

        # Sort by impact
        sorted_deltas = sorted(deltas.items(), key=lambda x: abs(x[1]), reverse=True)
        m_names = [m.replace("_", " ") for m, _ in sorted_deltas[:10]]
        d_vals = [d for _, d in sorted_deltas[:10]]
        bar_colors = ["#e74c3c" if d > 0 else "#2ecc71" for d in d_vals]

        ax.barh(range(len(m_names)), d_vals, color=bar_colors, alpha=0.8)
        ax.set_yticks(range(len(m_names)))
        ax.set_yticklabels(m_names, fontsize=8)
        ax.set_xlabel("RPE change when muscle drops to 50%")
        ax.set_title(f"{ex_name.replace('_', ' ').title()}\n(baseline RPE={rpe_base:.1f})")
        ax.axvline(0, color="black", linewidth=0.5)
        ax.invert_yaxis()
        ax.grid(True, alpha=0.3, axis="x")

plt.suptitle("RPE Sensitivity (g) — Which Muscle Fatigue Affects RPE Most?", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- Chart: RPE vs Set Number Within Workout ---
# Does prediction accuracy degrade later in a workout?

# Group test data by position within workout (detect workout boundaries by >2h gap)
test_sequences_eval = build_user_sequences(test_df)

set_positions = []
abs_errors = []

model.eval()
with torch.no_grad():
    for seq in test_sequences_eval[:100]:  # sample 100 users for speed
        T = len(seq["exercise_idx"])
        ex = seq["exercise_idx"].unsqueeze(0).to(DEVICE)
        w = seq["weight"].unsqueeze(0).to(DEVICE)
        r = seq["reps"].unsqueeze(0).to(DEVICE)
        rpe = seq["rpe"].unsqueeze(0).to(DEVICE)
        dt = seq["delta_t"].unsqueeze(0).to(DEVICE)

        rpe_preds_s, _ = model.forward_with_mpc_history(ex, w, r, rpe, dt)

        # Detect workout boundaries (delta_t > 2h in original scale)
        dt_hours_orig = np.expm1(seq["delta_t"].numpy() * DT_SCALE)
        set_pos = 0
        for t in range(T):
            if t > 0 and dt_hours_orig[t] > 2.0:
                set_pos = 0  # new workout
            set_positions.append(set_pos)
            abs_errors.append(abs(rpe_preds_s[t] * 10.0 - seq["rpe"][t].item() * 10.0))
            set_pos += 1

set_positions = np.array(set_positions)
abs_errors = np.array(abs_errors)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# MAE by set position (cap at 30)
ax = axes[0]
max_pos = 30
pos_mae = []
pos_count = []
for p in range(max_pos):
    mask = set_positions == p
    if mask.sum() > 50:
        pos_mae.append(np.mean(abs_errors[mask]))
        pos_count.append(mask.sum())
    else:
        pos_mae.append(np.nan)
        pos_count.append(0)

ax.bar(range(max_pos), pos_mae, color="steelblue", alpha=0.7)
ax.set_xlabel("Set # Within Workout")
ax.set_ylabel("MAE (RPE)")
ax.set_title("Prediction Accuracy by Set Position in Workout")
ax.grid(True, alpha=0.3, axis="y")

# RPE error by actual RPE bucket
ax = axes[1]
rpe_buckets = np.arange(3, 11)
bucket_mae = []
bucket_counts = []
# Recollect with buckets
all_actual = np.array([seq["rpe"].numpy() * 10.0 for seq in test_sequences_eval[:100] for _ in range(1)]).flatten()
# Use the already collected test_preds/test_targets
for rpe_val in rpe_buckets:
    mask = (test_targets >= rpe_val - 0.5) & (test_targets < rpe_val + 0.5)
    if mask.sum() > 0:
        bucket_mae.append(np.mean(np.abs(test_preds[mask] - test_targets[mask])))
        bucket_counts.append(mask.sum())
    else:
        bucket_mae.append(0)
        bucket_counts.append(0)

ax2 = ax.twinx()
ax.bar(rpe_buckets, bucket_mae, color="coral", alpha=0.7, width=0.6)
ax2.plot(rpe_buckets, bucket_counts, "s-", color="gray", alpha=0.5, markersize=4)
ax2.set_ylabel("Sample count", color="gray")
ax.set_xlabel("Actual RPE")
ax.set_ylabel("MAE (RPE)")
ax.set_title("Prediction Accuracy by RPE Range")
ax.set_xticks(rpe_buckets)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

In [ ]:
# --- Chart: Cross-Exercise Fatigue Transfer Matrix ---
# If you do 4 sets of exercise A, how much does RPE increase on exercise B?
# This reveals learned cross-exercise interference patterns

model.eval()
transfer_exercises = ["bench_press", "ohp", "dips", "squat", "deadlift",
                      "barbell_row", "lat_pulldown", "bicep_curl", "tricep_pushdown"]

n_ex = len(transfer_exercises)
transfer_matrix = np.zeros((n_ex, n_ex))

with torch.no_grad():
    for i, ex_a in enumerate(transfer_exercises):
        for j, ex_b in enumerate(transfer_exercises):
            ei_a = EXERCISE_TO_IDX[ex_a]
            ei_b = EXERCISE_TO_IDX[ex_b]

            # Baseline: RPE of exercise B with fresh muscles
            mpc_fresh = torch.ones(1, NUM_MUSCLES, device=DEVICE)
            e_embed_b = model.exercise_embed(torch.tensor([ei_b], device=DEVICE))
            w_b = torch.tensor([0.4], device=DEVICE)
            r_b = torch.tensor([0.27], device=DEVICE)
            rpe_fresh = model.g_net(w_b, r_b, e_embed_b, mpc_fresh).item() * 10.0

            # Do 4 sets of exercise A first
            mpc = torch.ones(1, NUM_MUSCLES, device=DEVICE)
            all_m_idx = torch.arange(NUM_MUSCLES, device=DEVICE)
            all_m_embed = model.muscle_embed(all_m_idx).unsqueeze(0)
            E_dim = all_m_embed.shape[-1]
            e_embed_a = model.exercise_embed(torch.tensor([ei_a], device=DEVICE))

            for _ in range(4):
                inv = model.involvement[torch.tensor([ei_a], device=DEVICE)]
                e_emb_exp = e_embed_a.unsqueeze(1).expand(-1, NUM_MUSCLES, -1).reshape(-1, E_dim)
                w_exp = torch.tensor([0.4], device=DEVICE).unsqueeze(1).expand(-1, NUM_MUSCLES).reshape(-1)
                r_exp = torch.tensor([0.27], device=DEVICE).unsqueeze(1).expand(-1, NUM_MUSCLES).reshape(-1)
                rpe_exp = torch.tensor([0.8], device=DEVICE).unsqueeze(1).expand(-1, NUM_MUSCLES).reshape(-1)
                drop = model.f_net(w_exp, r_exp, rpe_exp, mpc.reshape(-1),
                                   e_emb_exp, all_m_embed.reshape(-1, E_dim)).reshape(1, NUM_MUSCLES)
                mpc = (mpc * (1.0 - inv * drop)).clamp(min=0.1)

            # Now measure RPE of exercise B with fatigued muscles
            rpe_fatigued = model.g_net(w_b, r_b, e_embed_b, mpc).item() * 10.0
            transfer_matrix[i, j] = rpe_fatigued - rpe_fresh

fig, ax = plt.subplots(figsize=(10, 8))
labels = [e.replace("_", " ") for e in transfer_exercises]
im = ax.imshow(transfer_matrix, cmap="YlOrRd", vmin=0)
ax.set_xticks(range(n_ex))
ax.set_yticks(range(n_ex))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel("Exercise B (tested after)")
ax.set_ylabel("Exercise A (4 sets done first)")
ax.set_title("Cross-Exercise Fatigue Transfer\n(RPE increase on B after 4 sets of A)")

# Annotate cells
for i in range(n_ex):
    for j in range(n_ex):
        val = transfer_matrix[i, j]
        color = "white" if val > transfer_matrix.max() * 0.6 else "black"
        ax.text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=8, color=color)

plt.colorbar(im, ax=ax, label="RPE increase", shrink=0.8)
plt.tight_layout()
plt.show()